Installations

In [1]:
import pandas as pd
import pytz
import re
import unicodedata
import nltk
import json
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\fatim\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\fatim\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\fatim\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


commnets

In [2]:
comments = pd.read_csv('instagram_comments.csv')
data = comments.loc[:,['post_id','created_at','username','text','parent_comment_id']]

Drop empty comments
Remove comments from pdamsuryasembada
Change date from epoch miliseconds
Remove taggings from comments

In [3]:
data = data[data['username'].str.contains('pdamsuryasembada') == False]

data = data.drop_duplicates(subset=['text'])

local_timezone = pytz.timezone('Asia/Jakarta')
data['created_at'] = pd.to_datetime(data['created_at'], unit='s')
data['created_at'] = data['created_at'].dt.tz_localize('UTC').dt.tz_convert(local_timezone)
data['created_at'] = data['created_at'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

def remove_tags(text):
    if not isinstance(text, str):
        return ""
    return re.sub(r'@\w[\w\.]*', '', text).strip()

data['text'] = data['text'].apply(remove_tags)

data = data.dropna(subset=["text"])
data = data[data["text"].astype(str).str.strip() != ""]

cleaning

In [4]:
PUNCT_TO_REMOVE = "!#$%&()*+,./:;<=>@[\\]^_{|}~`"
def remove_punctuation(text):
    """custom function to remove the punctuation"""
    return text.translate(str.maketrans('', '', PUNCT_TO_REMOVE))

data["text"] = data["text"].apply(lambda text: remove_punctuation(text))

def remove_emojis(text):
    """Removes standard unicode emojis from the text."""
    if not isinstance(text, str):
        return text

    emoji_pattern = re.compile(
        "["                     
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F6FF"
        u"\U0001F1E0-\U0001F1FF"
        u"\U0001F700-\U0001F77F"
        u"\U0001F780-\U0001F7FF"
        u"\U0001F800-\U0001F8FF"
        u"\U0001F900-\U0001F9FF"
        u"\U0001FA70-\U0001FAFF"
        u"\u2600-\u26FF"
        u"\u2700-\u27BF"
        u"\ufe0f"
        "]+",
        flags=re.UNICODE,
    )

    return emoji_pattern.sub(r'', text)

data['text'] = data['text'].apply(remove_emojis)
data['text'] = data['text'].str.strip()

def normalize_fancy_unicode(text):
    if not isinstance(text, str):
        return text

    normalized = unicodedata.normalize("NFKD", text)

    cleaned = "".join(
        ch for ch in normalized
        if not unicodedata.combining(ch)
    )
    return cleaned

data["text"] = data["text"].apply(normalize_fancy_unicode)

data = data.dropna(subset=["text"])
data = data[data["text"].astype(str).str.strip() != ""]

case folding

In [5]:
data["text"] = data["text"].str.lower()

tokem

In [6]:
data['text'] = data['text'].astype(str).apply(word_tokenize)

In [7]:
def is_all_numbers(tokens):
    return all(re.fullmatch(r'\d+', tok) for tok in tokens)

data = data[~data['text'].apply(is_all_numbers)]


def normalize_mixed_number_token(token):
    if re.fullmatch(r'\d+', token):
        return "<num>"
    
    parts = re.findall(r'\d+|[a-zA-Z]+', token)

    if len(parts) == 1:
        return token

    parts = ["<num>" if p.isdigit() else p for p in parts]

    return " ".join(parts)


def normalize_tokens(token_list):
    normalized = []
    for tok in token_list:
        if tok.isdigit():
            normalized.append("<num>")
        elif re.search(r'\d', tok):
            normalized.append(normalize_mixed_number_token(tok))
        else:
            normalized.append(tok)
    return normalized

data["text"] = data["text"].apply(normalize_tokens)

In [8]:
data_partial = data.head(1000).copy()
# data_partial.to_csv('partial_no_pdam4.csv', index=False)

formalize

In [9]:
with open("dictionary/dict_template3_doneig.json", "r", encoding="utf-8") as f:
    formal_dict = json.load(f)
def apply_formalization(tokens, formal_dict):
    new_tokens = []
    for tok in tokens:
        if tok in formal_dict:
            new_tokens.append(formal_dict[tok])
        else:
            new_tokens.append(tok)
    return new_tokens

data_partial["text"] = data_partial["text"].apply(lambda tokens: apply_formalization(tokens, formal_dict))

deletus stopwordus

In [10]:
with open('stopwords2.txt', 'r', encoding='utf-8') as f:
    custom_stopwords = set([line.strip() for line in f if line.strip()])

def remove_stopwords(tokens, stopword_set):
    return [tok for tok in tokens if tok not in stopword_set and tok.strip() != ""]

data_partial["text"] = data_partial["text"].apply(lambda tokens: remove_stopwords(tokens, custom_stopwords))

stemmming

In [11]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()
def stem_tokens(tokens):
    return [stemmer.stem(tok) for tok in tokens]
data_partial["text"] = data_partial["text"].apply(stem_tokens)


tfidf ig

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
data_partial['processed_text'] = data_partial['text'].apply(lambda tokens: " ".join(tokens))
data_partial = data_partial[data_partial['processed_text'].str.strip() != ""]
vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    ngram_range=(1,1),
    min_df=2,
    max_df=0.9,
    sublinear_tf=True
)

vectors = vectorizer.fit_transform(data_partial['processed_text'])
feature_names = vectorizer.get_feature_names_out()
vectors_dense = vectors.toarray()

d:\Kuliah\PA\pdam-scraper\scrape_instagram\venv\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [17]:
print(f"Vector shape: {vectors.shape}\n")
print(f"nonzero: {vectors.nnz}\n")
sparsity = vectors.nnz / (980 * 682)
print(f"Sparsity: {sparsity:.4f}\n")
row = vectors[5]
indices = row.indices
values = row.data

for i, v in zip(indices, values):
    print(feature_names[i], v)

Vector shape: (980, 682)

nonzero: 6333

Sparsity: 0.0095

air 0.15339012728355553
surabaya 0.33330857793449487
barat 0.3763538313834667
daerah 0.2789011113666172
ptc 0.5222789684740339
mati 0.3067288844891148
parah 0.38445808826431227
banget 0.362329785260035


In [18]:
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.metrics import pairwise_distances
import numpy as np

Z = linkage(vectors_dense, method='centroid', metric='euclidean')
merge_distances = Z[:, 2]
v = np.abs(np.diff(merge_distances))
valleys = []
for i in range(1, len(v) - 1):
    if v[i] < v[i - 1] and v[i] < v[i + 1]:
        valleys.append(i)

if len(valleys) == 0:
    best_stage = np.argmin(v)
else:
    depths = []
    for i in valleys:
        depth = (v[i - 1] - v[i]) + (v[i + 1] - v[i])
        depths.append(depth)
    best_stage = valleys[np.argmax(depths)]

N = vectors_dense.shape[0]
k = N - best_stage

print(f"Optmial number of clusters = {k}")

cluster_labels = fcluster(Z, t=k, criterion='maxclust')

data_partial = data_partial.reset_index(drop=True)
data_partial['cluster'] = cluster_labels

print(data_partial[['processed_text', 'cluster']].head())

Optmial number of clusters = 532
                                      processed_text  cluster
0  selamat ulang hari surya moga sumber informasi...       20
1                        jalin kerjasama media keren       20
2                         selamat semangat indonesia      489
3                                 dm ku ga balas kak       20
4                                  salam tim spi min      411
